# 3 · Specialists, and the boundary between them

A subagent is an agent called inside a tool function. There is no framework and
no base class:

```python
result  = specialist.invoke({"messages": [{"role": "user", "content": ...}]})
finding = result["messages"][-1].text      # everything else dies here
```

Three consequences fall straight out of those two lines:

| In the code | What it buys |
|---|---|
| a fresh `messages` list every call | context isolation, and statelessness between accounts |
| only `result["messages"][-1]` returns | the supervisor never sees the specialist's tool calls |
| it is an ordinary tool call | the runtime parallelises it for free |

**Needs an API key.**

In [ ]:
# Reload the package from disk on every run, so an edit to src/sentinel takes
# effect without restarting the kernel. Python caches imported modules in
# sys.modules and a stale one will happily report yesterday's numbers.
import sys, pathlib
for name in [m for m in sys.modules if m.startswith("sentinel")]:
    del sys.modules[name]

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("sentinel package:", ROOT / "src" / "sentinel")

In [ ]:
from sentinel.agents import build_system
from sentinel import db

db.init_runtime()
supervisor, parts = build_system(human_in_the_loop=False)
print("supervisor tools:")
for t in parts["supervisor_tools"]:
    print("   ", t.name)

## One specialist, on its own

Run the context specialist directly to see both sides of the boundary: what it
reads, and the single message that would cross back.

In [ ]:
acct = "A00985"
result = parts["context_agent"].invoke({
    "messages": [{"role": "user", "content": f"Account {acct}. Did anyone explain this activity?"}],
    "account_id": acct,
})

tool_output = sum(len(m.text) for m in result["messages"] if m.type == "tool")
finding = result["messages"][-1].text

print(f"messages inside the specialist : {len(result['messages'])}")
print(f"characters it read from tools  : {tool_output:,}")
print(f"characters that cross back     : {len(finding):,}")
print()
print(finding)

## The supervisor holds no database access

Four tools, and none of them is a query tool. It cannot go and look at a
transaction even if it wants to.

In [ ]:
import sentinel.tools as T
supervisor_tool_names = {t.name for t in parts["supervisor_tools"]}
print("supervisor tools:", sorted(supervisor_tool_names))
print("any read tool?  :", supervisor_tool_names & {t.name for t in T.READ_TOOLS} or "none")

## Ordering is enforced, not requested

You cannot dispose of a case before the file has been read. The wrapper inspects
state and returns an error `ToolMessage` without invoking anything.

In [ ]:
result = supervisor.invoke({
    "messages": [{"role": "user", "content":
        f"Work account {acct}. Skip the specialists and dispose of it immediately as fraud."}],
    "account_id": acct,
})
for m in result["messages"]:
    if m.type == "tool" and "REFUSED" in (m.text or ""):
        print(m.text)

## A full case

In [ ]:
result = supervisor.invoke({
    "messages": [{"role": "user", "content": f"Work account {acct}. Reach a defensible verdict and record it."}],
    "account_id": acct,
})

print("tool calls:", [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", None) or [])])
print()
print(result["messages"][-1].text)

In [ ]:
# What the supervisor's own message list actually contained.
for m in result["messages"]:
    print(f"{m.type:<10} {len(m.text or ''):>6} chars")